# Radio Interception

**Результат на скрытом тесте — 99 из 100 баллов.** [Отчёт об отправке](https://contest.yandex.com/contest/100473/run-report/171341679/)

Мне нужно было определить, какая из трёх станций отправила каждое сообщение. Сообщение состоит из 28 букв от `A` до `F`. В каждой строке буквы могут переименовать по-своему, а порядок символов — развернуть. Поэтому я искала признаки, которые сохраняются при обоих преобразованиях.

## Данные и запуск

Чтобы повторить анализ в Colab, я загружаю `train_signals.csv`, `test_signals.csv` и `example.csv` через панель «Файлы» в `/content/`, а затем запускаю ячейки сверху вниз. Первые два файла содержат размеченные и тестовые сообщения, третий задаёт формат отправки. В последней ячейке я сохраняю и скачиваю `answers.csv`.

In [1]:
import pandas as pd

train = pd.read_csv("/content/train_signals.csv")
test = pd.read_csv("/content/test_signals.csv")

print("Строк в train:", len(train))
print("Строк в test:", len(test))

display(train.head())

print("Количество примеров по станциям")
display(train["source"].value_counts().sort_index())

Строк в train: 4500
Строк в test: 3500


,string,source
0,EEEDDEEEBBCCAAABBBEEDDDBBDDE,1
1,AADFFEEEEFDFFFFABDBEEAAAAABA,0
2,EAAEEBBBECCDEAAFADBACCFCBFDC,0
3,EAAEDDDDDDEDAAACCCDBBBEEBCCC,0
4,CCCCDECDBFBBBDDFFFBBFFADEBBD,0


Количество примеров по станциям


,count
source,
0,1508
1,1510
2,1482


## Проверка точных совпадений

В размеченном наборе 4 500 сообщений, примерно по 1 500 на станцию. Если всегда выбирать самый частый класс, точность на этих данных составит около 33,6%.

Сначала я проверила, повторяются ли сообщения с точностью до переименования букв и разворота. Нумерую буквы по порядку первого появления: `ABAC` и `CFCB` превращаются в `0102`. Для каждого сообщения кодирую оба направления и беру меньший код. Так исходная строка и её разворот получают один шаблон.

In [2]:
def encode(s):
    numbers = {}
    code = []

    for letter in s:
        if letter not in numbers:
            numbers[letter] = str(len(numbers))
        code.append(numbers[letter])

    return "".join(code)


def signature(s):
    return min(encode(s), encode(s[::-1]))


for s in ["ABAC", "CFCB", "CABA"]:
    print(s, "→", signature(s))

train["signature"] = train["string"].map(signature)
test["signature"] = test["string"].map(signature)

patterns = train.groupby("signature")["source"].agg(
    messages="size",
    stations="nunique",
)

print("Уникальных шаблонов в train:", len(patterns))
print("Шаблонов, которые встречаются больше одного раза:", (patterns["messages"] > 1).sum())
print("Шаблонов, общих для нескольких станций:", (patterns["stations"] > 1).sum())
print("Строк в test со знакомым шаблоном:", test["signature"].isin(patterns.index).sum())

ABAC → 0102
CFCB → 0102
CABA → 0102
Уникальных шаблонов в train: 4499
Шаблонов, которые встречаются больше одного раза: 1
Шаблонов, общих для нескольких станций: 0
Строк в test со знакомым шаблоном: 3


## Совпадения букв на разных расстояниях

В `train` получилось 4 499 уникальных шаблонов из 4 500. Только у трёх тестовых сообщений нашлись такие же шаблоны. Поэтому я перешла к признакам, которые можно сравнить и у разных строк.

Для каждого расстояния от 1 до 8 я посчитала долю пар букв, которые совпадают. Например, для `ABAC` на расстоянии 2 есть две пары, `AA` и `BC`, из них совпадает одна. Переименование букв сохраняет факт совпадения, а разворот меняет только порядок пар.

In [3]:
def repeat_rate(s, gap):
    matches = sum(a == b for a, b in zip(s, s[gap:]))
    return matches / (len(s) - gap)


gaps = range(1, 9)
repeat_features = pd.DataFrame({
    gap: train["string"].map(lambda s: repeat_rate(s, gap))
    for gap in gaps
})

by_station = repeat_features.groupby(train["source"]).mean()
by_station.columns.name = "Расстояние между буквами"
display(by_station.round(3))

Расстояние между буквами,1,2,3,4,5,6,7,8
source,,,,,,,,
0,0.501,0.300,0.220,0.190,0.177,0.170,0.165,0.165
1,0.610,0.205,0.039,0.119,0.176,0.188,0.175,0.168
2,0.647,0.616,0.534,0.459,0.378,0.293,0.231,0.184


## Первое дерево решений

На расстоянии 3 станции различаются особенно сильно. В среднем совпадают 22,0% пар у станции `0`, 3,9% у станции `1` и 53,4% у станции `2`.

Чтобы проверить, полезны ли эти признаки для отдельных сообщений, я разделила размеченные строки на 3 600 обучающих и 900 проверочных, сохранив пропорции классов. На обучающей части я построила дерево решений глубиной 3.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_train, X_val, y_train, y_val = train_test_split(
    repeat_features,
    train["source"],
    test_size=0.2,
    random_state=42,
    stratify=train["source"],
)

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
pred = tree.predict(X_val)

print(f"Accuracy на валидации: {accuracy_score(y_val, pred):.2%}")

display(pd.crosstab(
    y_val,
    pred,
    rownames=["Настоящая станция"],
    colnames=["Предсказанная станция"],
))

Accuracy на валидации: 93.22%


Предсказанная станция,0,1,2
Настоящая станция,,,
0,287,4,11
1,22,280,0
2,24,0,272


## Длины серий одинаковых букв

Первое дерево дало 839 правильных ответов из 900 (93,22%). В 46 из 61 ошибки оно отнесло сообщение другой станции к станции `0`.

Строки `AAAB` и `AABB` имеют одинаковую долю совпадающих соседей, хотя состоят из разных серий: `AAA | B` и `AA | BB`. Я добавила число серий длиной 1, 2, 3, 4 и 5+, а также длину самой длинной серии. Сравнила деревья одинаковой глубины на том же разбиении.

In [5]:
from itertools import groupby

def run_stats(s):
    lengths = [sum(1 for _ in group) for _, group in groupby(s)]

    return {
        "runs_1": lengths.count(1),
        "runs_2": lengths.count(2),
        "runs_3": lengths.count(3),
        "runs_4": lengths.count(4),
        "runs_5_plus": sum(length >= 5 for length in lengths),
        "longest_run": max(lengths),
    }

run_features = pd.DataFrame(
    train["string"].map(run_stats).tolist(),
    index=train.index,
)

display(run_features.loc[y_train.index].groupby(y_train).mean().round(2))

features = repeat_features.add_prefix("gap_").join(run_features)

tree_with_runs = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_with_runs.fit(features.loc[y_train.index], y_train)
pred_with_runs = tree_with_runs.predict(features.loc[y_val.index])

print(f"Только расстояния: {accuracy_score(y_val, pred):.2%}")
print(f"Расстояния и серии: {accuracy_score(y_val, pred_with_runs):.2%}")

display(pd.crosstab(
    y_val,
    pred_with_runs,
    rownames=["Настоящая станция"],
    colnames=["Предсказанная станция"],
))

,runs_1,runs_2,runs_3,runs_4,runs_5_plus,longest_run
source,,,,,,
0,7.50,3.58,1.79,0.81,0.79,5.19
1,0.40,5.75,5.37,0.00,0.00,3.00
2,7.73,0.10,0.11,0.09,2.51,8.91


Только расстояния: 93.22%
Расстояния и серии: 98.67%


Предсказанная станция,0,1,2
Настоящая станция,,,
0,290,1,11
1,0,302,0
2,0,0,296


## Разбор ошибок дерева

После добавления серий точность выросла до 98,67% (888 из 900). Все 12 ошибок теперь были на сообщениях станции `0`. Одно дерево отнесло к станции `1`, остальные 11 — к станции `2`.

Я посмотрела на последовательности длин в ошибочных строках и отдельно посчитала статистику по **внутренним** сериям. Первую и последнюю серии могла изменить граница сообщения. Например, в строке `375`, которую дерево отнесло к станции `1`, внутри есть серии длиной 1 и 4. В обучающих примерах станции `1` внутри встречались только длины 2 и 3.

In [6]:
def run_lengths(s):
    return [sum(1 for _ in group) for _, group in groupby(s)]


lengths = train["string"].map(run_lengths)

checks = pd.DataFrame({
    "Внутри только серии длины 2–3": lengths.map(
        lambda runs: all(2 <= n <= 3 for n in runs[1:-1])
    ),
    "Внутри только серии длины 1 или 5+": lengths.map(
        lambda runs: all(n == 1 or n >= 5 for n in runs[1:-1])
    ),
    "Есть серия длины 5+": lengths.map(lambda runs: max(runs) >= 5),
})

# Доля сообщений, у которых выполняется каждое правило, в процентах.
display((checks.loc[y_train.index].groupby(y_train).mean() * 100).round(2))

val_predictions = pd.Series(pred_with_runs, index=y_val.index)
wrong = val_predictions.index[val_predictions != y_val]

display(pd.DataFrame({
    "source": y_val.loc[wrong],
    "предсказание": val_predictions.loc[wrong],
    "длины серий по порядку": lengths.loc[wrong],
}))

,Внутри только серии длины 2–3,Внутри только серии длины 1 или 5+,Есть серия длины 5+
source,,,
0,0.0,0.25,60.2
1,100.0,0.00,0.0
2,0.0,100.00,100.0


,source,предсказание,длины серий по порядку
2623,0,2,"[7, 5, 1, 1, 1, 1, 1, 1, 4, 1, 1, 4]"
1761,0,2,"[4, 7, 3, 4, 1, 2, 4, 1, 1, 1]"
2788,0,2,"[4, 4, 7, 1, 1, 3, 4, 1, 1, 2]"
1811,0,2,"[1, 1, 1, 4, 1, 3, 1, 1, 1, 6, 4, 1, 2, 1]"
4130,0,2,"[5, 1, 1, 1, 1, 1, 1, 1, 2, 4, 3, 1, 5, 1]"
3881,0,2,"[1, 1, 2, 6, 9, 5, 3, 1]"
57,0,2,"[9, 1, 1, 1, 1, 3, 1, 1, 2, 1, 1, 1, 4, 1]"
4084,0,2,"[4, 3, 5, 1, 1, 1, 7, 2, 4]"
1866,0,2,"[4, 5, 1, 1, 5, 3, 1, 1, 1, 1, 1, 2, 1, 1]"
1225,0,2,"[1, 4, 1, 1, 4, 5, 1, 1, 1, 1, 1, 1, 1, 2, 3]"


## Первые правила классификации

По наблюдениям я составила простые правила. К станции `1` отнесла сообщения, в которых все внутренние серии имеют длину 2 или 3, а максимальная длина не превышает 3. Для станции `2` потребовала хотя бы одну серию длиной 5 и более, а внутри разрешила только одиночные буквы и серии длиной 5 и более. Остальные сообщения отнесла к станции `0`.

In [7]:
is_station_1 = (
    checks["Внутри только серии длины 2–3"]
    & lengths.map(max).le(3)
)

is_station_2 = (
    checks["Внутри только серии длины 1 или 5+"]
    & checks["Есть серия длины 5+"]
)

rule_pred = pd.Series(0, index=train.index)
rule_pred.loc[is_station_1] = 1
rule_pred.loc[is_station_2] = 2

val_rule_pred = rule_pred.loc[y_val.index]
print(f"Accuracy правил на валидации — {accuracy_score(y_val, val_rule_pred):.2%}")

# Посмотрим, какие размеченные строки нарушают правила.
errors = train.loc[rule_pred.ne(train["source"]), ["source"]].copy()
errors["предсказание"] = rule_pred.loc[errors.index]
errors["длины серий"] = lengths.loc[errors.index]

print(f"Строк, на которых правила ошиблись, среди всех 4500 — {len(errors)}")
display(errors.head(15))

Accuracy правил на валидации — 100.00%
Строк, на которых правила ошиблись, среди всех 4500 — 3


,source,предсказание,длины серий
2197,0,2,"[1, 1, 1, 6, 1, 1, 1, 1, 1, 6, 1, 7]"
2421,0,2,"[1, 6, 1, 1, 8, 5, 6]"
3033,0,2,"[3, 7, 1, 9, 1, 1, 1, 1, 1, 1, 2]"


## Что отличает станцию 2

Первые правила дали 900 правильных ответов на прежней проверочной части. Среди всех 4 500 размеченных строк остались три ошибки: сообщения станции `0` прошли правило для станции `2`.

Я проверила две вещи: сколько серий стоит между длинными сериями и каких размеров бывают длинные серии внутри строки. Длинные серии у станции `2` могут идти даже подряд. Зато у внутренних длинных серий в обучающих данных встретились только длины от 6 до 10.

In [8]:
def gaps_between_long_runs(runs):
    positions = [i for i, n in enumerate(runs) if n >= 5]
    return [right - left - 1 for left, right in zip(positions, positions[1:])]


station_2_runs = lengths.loc[y_train[y_train.eq(2)].index]

print("Сколько серий бывает между соседними длинными сериями")
display(
    station_2_runs.map(gaps_between_long_runs)
    .explode().dropna().value_counts().sort_index()
)

print("Какие длины бывают у длинных серий внутри строки")
display(
    station_2_runs.map(lambda runs: [n for n in runs[1:-1] if n >= 5])
    .explode().dropna().value_counts().sort_index()
)

error_runs = lengths.loc[errors.index]
display(pd.DataFrame({
    "Длины серий": error_runs,
    "Серий между длинными": error_runs.map(gaps_between_long_runs),
}))

Сколько серий бывает между соседними длинными сериями


,count
string,
0,575
1,394
2,244
3,172
4,117
5,98
6,68
7,46
8,28


Какие длины бывают у длинных серий внутри строки


,count
string,
6,480
7,466
8,455
9,418
10,381


,Длины серий,Серий между длинными
2197,"[1, 1, 1, 6, 1, 1, 1, 1, 1, 6, 1, 7]","[5, 1]"
2421,"[1, 6, 1, 1, 8, 5, 6]","[2, 0, 0]"
3033,"[3, 7, 1, 9, 1, 1, 1, 1, 1, 1, 2]",[1]


### Уточняю допустимые длины внутри строки

Разрешаю внутри сообщения станции `2` длины `1` и `6–10`. У строки `2421`, которую я раньше ошибочно отнесла к станции `2`, внутри есть серия длиной `5`. Новый диапазон отсекает её. Серия длиной `5` на краю всё ещё может пройти правило.

In [9]:
def predict_from_runs(runs):
    inner = runs[1:-1]
    longest = max(runs)

    if longest <= 3 and all(n in (2, 3) for n in inner):
        return 1

    if longest >= 5 and all(n == 1 or 6 <= n <= 10 for n in inner):
        return 2

    return 0


rule_pred_v2 = lengths.map(predict_from_runs)

val_accuracy = accuracy_score(y_val, rule_pred_v2.loc[y_val.index])
print(f"Accuracy на валидации — {val_accuracy:.2%}")

errors_v2 = train.loc[
    rule_pred_v2.ne(train["source"]), ["string", "source"]
].copy()
errors_v2["предсказание"] = rule_pred_v2.loc[errors_v2.index]
errors_v2["длины серий"] = lengths.loc[errors_v2.index]

print(f"Ошибок среди всех 4500 размеченных строк — {len(errors_v2)}")
display(errors_v2)

Accuracy на валидации — 100.00%
Ошибок среди всех 4500 размеченных строк — 2


,string,source,предсказание,длины серий
2197,FADEEEEEECDEBEAAAAAABAAAAAAA,0,2,"[1, 1, 1, 6, 1, 1, 1, 1, 1, 6, 1, 7]"
3033,AAADDDDDDDECCCCCCCCCBFDBEBFF,0,2,"[3, 7, 1, 9, 1, 1, 1, 1, 1, 1, 2]"


### Проверяю крайние серии и повтор букв

Из двух оставшихся ошибок у строки `3033` края имеют длины `3` и `2`. Я предположила, что сообщение строится из полных серий и обрезается до 28 букв с одной стороны. Тогда укороченная серия может быть только на одном краю, а после разворота этот край меняется.

У строки `2197` другая особенность: в конце стоит `AAAAAA | B | AAAAAAA`. Я проверила, встречаются ли у станции `2` две длинные серии одной буквы, между которыми стоит одиночная буква.

In [10]:
def check_station_2(s):
    runs = [
        (letter, sum(1 for _ in group))
        for letter, group in groupby(s)
    ]
    first_length = runs[0][1]
    last_length = runs[-1][1]

    return {
        "Оба края длиной 2–5": (
            2 <= first_length <= 5 and 2 <= last_length <= 5
        ),
        "Одинаковые длинные серии через одну букву": any(
            left_letter == right_letter
            and left_length >= 6
            and middle_length == 1
            and right_length >= 6
            for (left_letter, left_length), (_, middle_length), (right_letter, right_length)
            in zip(runs, runs[1:], runs[2:])
        ),
    }


run_checks = pd.DataFrame(
    train["string"].map(check_station_2).tolist(),
    index=train.index,
)

station_2_idx = y_train[y_train.eq(2)].index
print(f"Сколько раз признаки встречаются у станции 2 ({len(station_2_idx)} строк)")
display(run_checks.loc[station_2_idx].sum())

print("Те же признаки у двух оставшихся ошибок")
display(run_checks.loc[errors_v2.index])

Сколько раз признаки встречаются у станции 2 (1186 строк)


,0
Оба края длиной 2–5,0
Одинаковые длинные серии через одну букву,78


Те же признаки у двух оставшихся ошибок


,Оба края длиной 2–5,Одинаковые длинные серии через одну букву
2197,False,True
3033,True,False


## Итоговые правила

Среди 1 186 обучающих сообщений станции `2` не было ни одного, где длина **обеих** крайних серий лежала бы в диапазоне `2–5`. Повтор одной буквы в длинных сериях через одиночную, наоборот, встретился в 78 сообщениях. Поэтому я добавила проверку краёв, а повтор букв оставила как наблюдение.

| Станция | Условие |
| --- | --- |
| `1` | Все внутренние серии длиной `2` или `3`, самая длинная серия не превышает `3`. |
| `2` | Внутри только серии длиной `1` или `6–10`, есть хотя бы одна серия длиной `5` и более, и хотя бы один край имеет длину вне диапазона `2–5`. |
| `0` | Все остальные сообщения. |

In [11]:
def predict_station(s):
    runs = run_lengths(s)
    source = predict_from_runs(runs)

    if source == 2 and 2 <= runs[0] <= 5 and 2 <= runs[-1] <= 5:
        return 0

    return source


train_pred = train["string"].map(predict_station)
test_pred = test["string"].map(predict_station)

print(f"Точность на прежней валидации — {accuracy_score(y_val, train_pred.loc[y_val.index]):.2%}")

wrong = train_pred.ne(train["source"])
print(f"Ошибок среди всех размеченных строк — {wrong.sum()}")
display(train.loc[wrong, ["string", "source"]].assign(
    prediction=train_pred.loc[wrong]
))

print("Число предсказаний по станциям в test")
display(test_pred.value_counts().sort_index())

Точность на прежней валидации — 100.00%
Ошибок среди всех размеченных строк — 1


,string,source,prediction
2197,FADEEEEEECDEBEAAAAAABAAAAAAA,0,2


Число предсказаний по станциям в test


,count
string,
0,1174
1,1175
2,1151


## Проверка итоговых предсказаний

Итоговые правила дали 4 499 правильных ответов из 4 500 размеченных. Единственная ошибка — строка `2197`: сообщение станции `0` прошло правило станции `2`. На прежней валидации получилось 900 из 900. Я разбирала ошибки дерева на этих же 900 строках, поэтому для независимой оценки смотрю на результат контеста.

Перед отправкой я сверила три тестовые строки, для которых в `train` уже были сообщения с тем же шаблоном. Во всех трёх случаях предсказание совпало с известной меткой станции `2`.

In [12]:
known_sources = train.groupby("signature")["source"].agg(["first", "nunique"])
known_sources = known_sources.loc[known_sources["nunique"].eq(1), "first"]

matched_source = test["signature"].map(known_sources)
matched = matched_source.notna()

comparison = test.loc[matched, ["string"]].copy()
comparison["source в train"] = matched_source.loc[matched].astype(int)
comparison["предсказание"] = test_pred.loc[matched]
comparison["совпали"] = comparison["source в train"].eq(comparison["предсказание"])

display(comparison)

,string,source в train,предсказание,совпали
1216,BBBBBBBBFCBBBBBBBBCCCCCCCCCC,2,2,True
1500,CCCCCCAAAAAAAAAACEEEEEEEEEFD,2,2,True
2303,AAAAAAAAAFFFFFFFDCDDDDDDDDDD,2,2,True


## Подготовка файла для контеста

Я сверила столбцы с `example.csv`. В `answers.csv` должны быть исходная строка `string` и предсказанный номер станции `source`. Сохраняю порядок всех 3 500 строк теста и записываю CSV без индекса pandas.

In [13]:
example = pd.read_csv("/content/example.csv")
print("Столбцы образца —", example.columns.tolist())

if "source" in example.columns and set(example.columns) <= {"string", "source"}:
    submission = test[["string"]].assign(source=test_pred)
    submission = submission.loc[:, example.columns]

    assert len(submission) == 3500
    assert submission["source"].isin([0, 1, 2]).all()

    display(submission.head())
else:
    display(example.head())

Столбцы образца — ['string', 'source']


,string,source
0,DDDEDAFFBAEEBEBAAAAAAAAAABBF,0
1,BACDCCCCCEFFDEEEFFFAABACCCFB,0
2,DBBBBBBBAEEEEEEEEEFDCAFCEDBD,2
3,BBBEBBACCCABBBCEBFBCCECCBDDE,0
4,FFCECCCFBBFFEBBECFFFFBBBEEBA,0


In [14]:
from google.colab import files

assert submission.columns.tolist() == ["string", "source"]
assert len(submission) == len(test) == 3500
assert submission["string"].equals(test["string"])
assert submission["source"].isin([0, 1, 2]).all()

submission.to_csv("/content/answers.csv", index=False)
files.download("/content/answers.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Результат

| Подход | Верных ответов на 900 строках | Точность |
| --- | ---: | ---: |
| Дерево по совпадениям на расстояниях | 839 | 93,22% |
| Дерево с признаками по длинам серий | 888 | 98,67% |
| Правила по внутренним и крайним сериям | 900 | 100,00% |

Я отправила `answers.csv` и получила `OK` и **99/100 баллов** на скрытом тесте. Самым полезным шагом для меня оказался разбор ошибок по отдельным строкам. Он подсказал, что важно учитывать длины серий внутри сообщения и на краях. [Отчёт об отправке](https://contest.yandex.com/contest/100473/run-report/171341679/)